# Sizing the cmfb_sc2 Common-Mode Feedback

Adapted from `macros/ota_fcasc_fd/scripts/sizing/lookup_commands.ipynb`, which is the overview of the pygmid lookup commands this notebook uses.

**Copyright 2026 Michael Koefinger**

SPDX-License-Identifier: Apache-2.0 WITH SHL-2.1

cmfb_sc2 was sized by hand and simulation, so this notebook rebuilds the derivation after the fact. It starts from what the common-mode loop and the integrator ask of it, derives the capacitors and the switches, then reads the drawn sizes from the schematic and compares. Nothing here changes a schematic. Where the derivation disagrees with the drawing, the cell says so and leaves the drawing alone.

Every number below is a gm/ID model estimate from the tt tables at 300 K, with no mismatch, no process corner and no layout wiring unless a cell says otherwise. The simulated values are in `macros/cmfb_sc2/README.md` and `macros/sc_int2/README.md`. The README has no simulation of the common-mode loop on its own, so the settling of the loop is derived but not checked against a number. The tables hold no Pelgrom data, so capacitor and switch matching is left to the Monte Carlo.

The block is the Figure 7.11 style SC common-mode feedback. Two sense capacitors `C1`, `C3` run from `vo_p`, `vo_n` to `vfb`, two refresh capacitors `C2`, `C4` are charged to `vcm - vbias` in phase 2 and put between the outputs and `vfb` in phase 1. Each cycle removes the share k = C2 / (C1 + C2) of the common-mode error. Six `sw_tgate_cmfb2` switches and two `inv_lv` make the phases. The only bias is the reference `vbias`, which is the OTA `vbn` and sets the precharge voltage. There is no current to size, so the sizing is capacitors, switches and the clock inverters.

In [1]:
# Packages
from pygmid import Lookup as lk
import numpy as np
import re
from pathlib import Path

In [2]:
# Spec and the choices the READMEs give
VDD, VCM, VBIAS = 1.5, 0.75, 0.285      # V, vbias is the OTA vbn (ota_fcasc2 README, tt)
FCLK     = 1.024e6                      # Hz
T_HALF   = 0.5 / FCLK                   # s, 488 ns
T_GAP    = 3.24e-9                      # s, non-overlap between the phases (dsm2 README, shortest)
T_PH     = T_HALF - T_GAP               # s, one phase
C_UM2    = 1.287                        # fF per um2, cap_cmomf model
REAL     = {'sense': 96.0 / 131.9, 'refresh': 42.64 / 63.4}   # estimated real over model, cmfb_sc2 README (field solve scaled fit)
SETTLE   = (0.01, 20)                   # CM offset left after N cycles, assumed, the README has no spec for it
SETTLE_ALT = (0.01, 15)                 # the same with a tighter start-up
V_NOISE  = 0.5e-3                       # V rms, kT/C of the refresh sample on vfb, assumed
V_CM_ERR = 20e-3                        # V, output common mode error allowed by the OTA swing (1.42 V), assumed
VFB      = VBIAS                        # V, at the fixed point vfb = avg(vo) + vbias - vcm = vbias
T_TARGET = 10                           # time constants of settling per phase
K_B_T    = 1.380649e-23 * 300
CELL     = 'cmfb_sc2'
DATA     = Path('data')
SCH      = Path('../schematic') / (CELL + '.sch')
SW_CDL   = Path('netlist/sw_tgate_cmfb2_klayout.cdl')
INV_CDL  = Path('netlist/inv_lv_klayout.cdl')
OTA_NET  = Path('netlist/ota_fcasc2_top_magic.spice')

### Read the sizes out of the schematic

The schematic is the source of truth, so nothing here restates a drawn size. The capacitor units are read from `cmfb_sc2.sch` (vector instances counted by their range). The switch and inverter devices come from the CDL exports of `sw_tgate_cmfb2` and `inv_lv`, which also give the gate connections, and the parasitic on `vfb` from the exported OTA netlist. The checks are the invariants the topology rests on: both sides equal, six switches, two inverters, one switch length.

In [3]:
def num(s):
    # 3e-6, 11.39e-6 or 1u, all returned in um
    return float(s[:-1]) if s.endswith('u') else float(s) * 1e6

def count(name):
    m = re.search(r'\[(\d+):(\d+)\]', name)
    return abs(int(m.group(1)) - int(m.group(2))) + 1 if m else 1

txt = SCH.read_text()
D = {}
for m in re.finditer(r'^C \{([^}]+)\.sym\} \S+ \S+ \S+ \S+ \{name=(\S+)\n(.*?)\}', txt, re.S | re.M):
    a = dict(re.findall(r'^(\w+)=(\S+)$', m.group(3), re.M))
    if m.group(1) == 'cap_cmomf':
        D[m.group(2)] = dict(w=num(a['w']), l=num(a['l']), n=count(m.group(2)))
N_SW, N_INV = len(re.findall(r'sw_tgate_cmfb2\.sym', txt)), len(re.findall(r'inv_lv\.sym', txt))

def read_mos(path, sub=None):
    # (type, d, g, s, W, L, m) per MOS of a CDL or SPICE netlist
    out = []
    for ln in path.read_text().splitlines():
        m = re.match(r'X?M\w+ (\S+) (\S+) (\S+) \S+ sg13_lv_([np])mos w=(\S+) l=(\S+)(?: ng=\d+)?(?: m=(\d+))?', ln)
        if m: out.append((m.group(4), m.group(1), m.group(2), m.group(3), num(m.group(5)), num(m.group(6)), int(m.group(7) or 1)))
    return out

SW, INV = read_mos(SW_CDL), read_mos(INV_CDL)
for n in D:
    print(f"{n:15s} {D[n]['n']:2d} x {D[n]['w']:.1f} um x {D[n]['l']:.3f} um")
print(f'{N_SW} switches, {N_INV} clock inverters, sense {D["C1[2:0]"]["n"]} units, refresh {D["C2[3:0]"]["n"]} units')
print('switch devices (type, W, L):', [(t, w, l) for t, d, g, s, w, l, m in SW if g not in ('vss', 'vdd')])
print('inverter devices (type, W, L):', [(t, w, l) for t, d, g, s, w, l, m in INV])

assert D['C1[2:0]'] == D['C3[2:0]'] and D['C2[3:0]'] == D['C4[3:0]'], 'the two sides differ'
assert (N_SW, N_INV) == (6, 2)
assert {l for t, d, g, s, w, l, m in SW} == {1.0}, 'mixed switch lengths'

C2[3:0]          4 x 3.0 um x 4.105 um
C4[3:0]          4 x 3.0 um x 4.105 um
C1[2:0]          3 x 3.0 um x 11.390 um
C3[2:0]          3 x 3.0 um x 11.390 um
CDUM_COL[13:0]  14 x 3.0 um x 2.000 um
6 switches, 2 clock inverters, sense 3 units, refresh 4 units
switch devices (type, W, L): [('n', 1.0, 1.0), ('p', 3.0, 1.0), ('n', 0.5, 1.0), ('n', 0.5, 1.0), ('p', 1.5, 1.0), ('p', 1.5, 1.0)]
inverter devices (type, W, L): [('p', 2.0, 0.13), ('n', 1.0, 0.13)]


### Lookups

The conductance of the transmission gate is read as in `size_sw_tgate_cmfb2.ipynb`: `Ron = 1/GDS` at VDS = 10 mV, NMOS with its gate at VDD and its bulk at VSS, PMOS the other way round, over the switched potential. Where a device is off the interpolated `GDS` goes slightly negative at the sweep ends and is clamped at zero.

In [4]:
nm = lk(str(DATA / 'sg13cmos5l_lv_nmos.mat'))
pm = lk(str(DATA / 'sg13cmos5l_lv_pmos.mat'))
T = {'n': nm, 'p': pm}

def ext(t, f, L, vgs, vds, vsb=0.0, w=1.0):
    return float(t.lookup(f, L=L, VGS=vgs, VDS=vds, VSB=vsb)) / float(t['W']) * w

def ron_tg(wn, wp, L, vs, vds=0.01):
    # transmission gate between the node at vs and a node vds away, both devices at full gate drive
    gn = np.maximum(np.diag(nm.lookup('GDS', L=L, VGS=VDD - vs, VDS=vds, VSB=vs)), 0) / nm['W']
    gp = np.maximum(np.diag(pm.lookup('GDS', L=L, VGS=vs, VDS=vds, VSB=VDD - vs)), 0) / pm['W']
    return 1 / (wn * gn + wp * gp).clip(1e-12)

def unit_c(name):
    return D[name]['w'] * D[name]['l'] * C_UM2 * 1e-15

### Capacitors

The unit is `cap_cmomf` with the model density of 1.287 fF/um2. The estimated real values are the README ones (the model reads about 1.37 times high for the sense unit and 1.49 times for the refresh unit), they are not derived here. The simulations use the model value.

The loop removes k = C2 / (C1 + C2) of the common-mode error per cycle, 0.325 with the model and 0.308 with the estimated real values. The README gives 31 %. How much k the loop needs depends on the start-up and on the disturbance, and the README gives no number, so the cell assumes the common mode has to be within 1 % after 20 cycles, and after 15 for a tighter start. That needs k of 0.206 and 0.264, which is 2.3 and 3.2 units of the refresh cap, so 3 and 4 units. The drawn 4 units agree with the tighter start and are one above the 20 cycle assumption.

The noise of the refresh sample is kT/C2. For a limit of 0.5 mV on `vfb` the refresh cap has to be 16.6 fF real, 1.6 units, so the 4 units (42.6 fF, 0.31 mV) have margin. Before 2026-10-20 the refresh cap was 1 unit with k = 0.10 and 44 cycles to 1 %, the move to 4 units brings that to 12.5 cycles.

In [5]:
C1u, C2u = unit_c('C1[2:0]'), unit_c('C2[3:0]')
N1, N2 = D['C1[2:0]']['n'], D['C2[3:0]']['n']
c1m, c2m = N1 * C1u, N2 * C2u
c1r, c2r = c1m * REAL['sense'], c2m * REAL['refresh']
print(f'sense   C1, C3: {N1} x {C1u*1e15:.2f} fF = {c1m*1e15:.1f} fF model, {c1r*1e15:.1f} fF estimated real')
print(f'refresh C2, C4: {N2} x {C2u*1e15:.2f} fF = {c2m*1e15:.1f} fF model, {c2r*1e15:.1f} fF estimated real')
print(f'sense over refresh {c1m/c2m:.2f} model, {c1r/c2r:.2f} real')

# charge sharing per cycle, two sense and two refresh caps on vfb
k = lambda cs, cr: cr / (cs + cr)
print(f'share of the error removed per cycle k = C2 / (C1 + C2): {k(c1m, c2m):.3f} model, {k(c1r, c2r):.3f} real')
for tag, (eps, n) in (('settle', SETTLE), ('tight', SETTLE_ALT)):
    kmin = 1 - eps ** (1 / n)
    print(f'{tag}: {eps*100:.0f} % left after {n} cycles needs k >= {kmin:.3f}, C2 >= {kmin/(1-kmin)*c1r*1e15:.1f} fF real = {kmin/(1-kmin)*c1r/(C2u*REAL["refresh"]):.2f} units')
print('units  k real  cycles to 1 %  sqrt(kT/C2) / mV')
for u in range(1, 7):
    cr = u * C2u * REAL['refresh']
    kk = k(c1r, cr)
    print(f'{u:5d} {kk:7.3f} {np.log(0.01)/np.log(1-kk):13.1f} {np.sqrt(K_B_T/cr)*1e3:15.2f}')
u_noise = V_NOISE ** -2 * K_B_T / (C2u * REAL['refresh'])
print(f'noise limit {V_NOISE*1e3:.1f} mV needs C2 >= {K_B_T/V_NOISE**2*1e15:.1f} fF real = {u_noise:.2f} units')

sense   C1, C3: 3 x 43.98 fF = 131.9 fF model, 96.0 fF estimated real
refresh C2, C4: 4 x 15.85 fF = 63.4 fF model, 42.6 fF estimated real
sense over refresh 2.08 model, 2.25 real
share of the error removed per cycle k = C2 / (C1 + C2): 0.325 model, 0.308 real
settle: 1 % left after 20 cycles needs k >= 0.206, C2 >= 24.9 fF real = 2.33 units
tight: 1 % left after 15 cycles needs k >= 0.264, C2 >= 34.5 fF real = 3.24 units
units  k real  cycles to 1 %  sqrt(kT/C2) / mV
    1   0.100          43.7            0.62
    2   0.182          23.0            0.44
    3   0.250          16.0            0.36
    4   0.308          12.5            0.31
    5   0.357          10.4            0.28
    6   0.400           9.0            0.25
noise limit 0.5 mV needs C2 >= 16.6 fF real = 1.55 units


### Sense capacitors and the load on the OTA

At the fixed point of the loop `vfb` sits at `vbias` (vfb = avg(vo) + vbias - vcm with avg(vo) = vcm), 0.285 V. It is the gate of the two OTA current sinks, 8 um of NMOS at L = 3 um in total, read from the exported OTA netlist. The table gives 148 fF for them at that voltage (L = 3 um is not on the table grid, so the lookup is interpolated). The sense capacitors divide against that parasitic, so a common-mode step at the outputs reaches `vfb` as 2 C1 / (2 C1 + Cpar), 0.57 with the drawn 3 units, 0.68 with 5. There is no spec for this in the READMEs. It halves the gain of the common-mode loop, which the OTA gain makes up, and it is the reason a larger refresh cap would not help.

The sense cap also sits on each OTA output permanently, 96 fF real, a quarter of the 12-unit integrator feedback capacitor. That is the cost of every added unit on the OTA settling. The drawn 3 units are the same unit as the integrator capacitors, which the layout needs.

In [6]:
# vfb is the gate of the OTA current sinks, read from the exported OTA netlist
ota = read_mos(OTA_NET)
w_vfb = sum(w * m for t, d, g, s, w, l, m in ota if g == 'vfb')
l_vfb = {l for t, d, g, s, w, l, m in ota if g == 'vfb'}.pop()
cpar = abs(ext(nm, 'CGG', l_vfb, VFB, 0.3, 0.0, w_vfb)) + sum(abs(ext(nm, 'CDD', l, 0.0, VFB, 0.0, w * m)) for t, d, g, s, w, l, m in ota if d == 'vfb' or s == 'vfb')
print(f'gates on vfb: {w_vfb:g} um in total at L = {l_vfb:g} um (not on the table grid, interpolated), parasitic {cpar*1e15:.1f} fF')
print('sense units  C1 real/fF  attenuation 2 C1 / (2 C1 + Cpar)')
for u in range(1, 6):
    c = u * C1u * REAL['sense']
    print(f'{u:11d} {c*1e15:11.1f} {2*c/(2*c+cpar):12.3f}')
c_load = 2 * c1r
print(f'load of the sense caps on each OTA output {c1r*1e15:.0f} fF real, {c1r/(12*32e-15)*100:.0f} % of the integrator feedback cap (12 x 32 fF)')

gates on vfb: 8 um in total at L = 3 um (not on the table grid, interpolated), parasitic 147.5 fF
sense units  C1 real/fF  attenuation 2 C1 / (2 C1 + Cpar)
          1        32.0        0.303
          2        64.0        0.465
          3        96.0        0.566
          4       128.0        0.634
          5       160.0        0.684
load of the sense caps on each OTA output 96 fF real, 25 % of the integrator feedback cap (12 x 32 fF)


### Switches

The switch is `Wn` 1u, `Wp` 3u, L 1u with half-width dummies. The refresh capacitor is charged through two switches in series and transfers through two, so the time constant is about 2 Ron C. The worst Ron over the potential range is 5560 ohm at 1.1 V, the figure of `size_sw_tgate_cmfb2.ipynb`. At the potentials the switch sees it is lower: 3.1 kOhm at `vbias` and `vfb` (0.285 V) and 4.5 kOhm at `vcm`. With the worst Ron and the model capacitance the precharge tau is 0.70 ns and the transfer tau 0.57 ns, so a phase of 485 ns holds 688 time constants. Ron alone would allow the smallest device on the grid (0.5u/1.5u). The 1u/3u is not set by Ron.

The charge moved per cycle is 29.5 fC per side (63.4 fF at 0.465 V).

In [7]:
SWN = max(w for t, d, g, s, w, l, m in SW if t == 'n' and g == 'g')
SWP = max(w for t, d, g, s, w, l, m in SW if t == 'p' and g == 'gb')
VS = np.arange(0.2, 1.41, 0.01)
ron = ron_tg(SWN, SWP, 1.0, VS)
k_ = int(np.argmax(ron))
print(f'switch {SWN:g}u / {SWP:g}u at L = 1u: worst {ron[k_]:.0f} ohm at {VS[k_]:.2f} V')
for tag, v in (('vbias, vfb', VBIAS), ('vcm', VCM)):
    print(f'  at {tag:5s} {v:.3f} V: {ron_tg(SWN, SWP, 1.0, np.array([max(v, 0.2)] * 2))[0]:.0f} ohm')

# precharge through two switches to vcm and vbias, transfer through two switches onto vfb
r_w = ron[k_]
for tag, c in (('model', c2m), ('real', c2r)):
    tau_pre = 2 * r_w * c
    tau_tr = 2 * r_w * (c * 2 * c1m * (REAL['sense'] if tag == 'real' else 1) / (c + 2 * c1m * (REAL['sense'] if tag == 'real' else 1)))
    print(f'{tag:5s}: precharge tau {tau_pre*1e9:.2f} ns, transfer tau {tau_tr*1e9:.2f} ns, {T_PH/max(tau_pre, tau_tr):.0f} tau in a phase of {T_PH*1e9:.0f} ns')

# smallest Wn on the 0.5u grid with Wp = 3 Wn that settles in T_TARGET time constants
wn = 0.5
while 2 * ron_tg(wn, 3 * wn, 1.0, VS).max() * c2m * T_TARGET > T_PH: wn += 0.5
print(f'Ron alone needs only Wn {wn:g}u / Wp {3*wn:g}u for {T_TARGET} tau in a phase (the smallest on the grid)')
q = c2m * (VCM - VBIAS)
print(f'refresh charge per cycle and side: {c2m*1e15:.1f} fF x {VCM-VBIAS:.3f} V = {q*1e15:.1f} fC model')

switch 1u / 3u at L = 1u: worst 5560 ohm at 1.10 V
  at vbias, vfb 0.285 V: 3129 ohm
  at vcm   0.750 V: 4536 ohm
model: precharge tau 0.70 ns, transfer tau 0.57 ns, 688 tau in a phase of 485 ns
real : precharge tau 0.47 ns, transfer tau 0.39 ns, 1023 tau in a phase of 485 ns
Ron alone needs only Wn 0.5u / Wp 1.5u for 10 tau in a phase (the smallest on the grid)
refresh charge per cycle and side: 63.4 fF x 0.465 V = 29.5 fC model


### Leakage and the common-mode error

A wider switch leaks more. The off current from the table (tt, VDS = 0.4 V, gates at the rails) is 0.30 nA for NMOS 1u plus PMOS 3u. Almost all of it is the NMOS, 0.3 nA/um, which the table gives at L = 0.5u, 1u and 5u alike, so it is the floor of the sweep and not a physical leakage. The PMOS reads 2 pA/um. As an upper bound the droop of `vfb` over a half period is 0.41 mV on 363 fF (two sense caps and the parasitic). The loop removes k of it per cycle, so the standing common-mode error is the droop over k, 1.3 mV. The off current that would use a 20 mV limit is 4.6 nA, 15 times the table value, so leakage does not limit the width here either. It would at a limit of about 1 mV. The README sentence that the leakage matters far more than Ron holds for the integrator virtual ground in `sc_int2`, where L = 1u comes from, and is not what binds the CMFB. The leak bench of the sc_int2 README is the check.

In [8]:
# off current of one switch (NMOS 1u + PMOS 3u, gates at the rails), table value at tt, an upper bound
ioff = (ext(nm, 'ID', 1.0, 0.0, 0.4, 0.0, SWN) + ext(pm, 'ID', 1.0, 0.0, 0.4, 0.0, SWP))
c_vfb = 2 * c1r + cpar
dv_cyc = ioff * T_HALF / c_vfb
print(f'off current {ioff*1e9:.2f} nA, droop of vfb in a half period {dv_cyc*1e3:.2f} mV on {c_vfb*1e15:.0f} fF')
print(f'common mode error = droop / k = {dv_cyc/k(c1r, c2r)*1e3:.1f} mV, limit {V_CM_ERR*1e3:.0f} mV')
i_max = k(c1r, c2r) * V_CM_ERR * c_vfb / T_HALF
print(f'off current that uses the limit: {i_max*1e9:.2f} nA')
for l in (0.5, 1.0, 5.0):
    print(f'  table at L = {l:g}u: NMOS {ext(nm, "ID", l, 0.0, 0.4, 0.0, 1.0)*1e12:.0f} pA/um, PMOS {ext(pm, "ID", l, 0.0, 0.4, 0.0, 1.0)*1e12:.0f} pA/um')

off current 0.30 nA, droop of vfb in a half period 0.44 mV on 340 fF
common mode error = droop / k = 1.4 mV, limit 20 mV
off current that uses the limit: 4.28 nA
  table at L = 0.5u: NMOS 247 pA/um, PMOS 3 pA/um
  table at L = 1u: NMOS 299 pA/um, PMOS 2 pA/um
  table at L = 5u: NMOS 194 pA/um, PMOS 0 pA/um


### Clock inverters

`inv_lv` is 1u NMOS and 2u PMOS at L = 0.13u. Each drives the complement gates of three switches, 147 fF. The dummies make the direct and the complement gate nets carry the same 4 um of gate per switch, so the two phase nets are loaded alike. The inverter delay is 123 ps, 4 % of the 3.24 ns gap, so the complement phase is late by that much at the switches against the direct phase. The inverter is not sized by this notebook, the delay only shows that it does not eat the gap.

In [9]:
# complement phase nets: gb of three switches, driven by inv_lv
gb_load = 3 * sum(abs(ext(T[t], 'CGG', l, 0.75, 0.0, 0.0, w)) for t, d, g, s, w, l, m in SW if g == 'gb')
g_load = 3 * sum(abs(ext(T[t], 'CGG', l, 0.75, 0.0, 0.0, w)) for t, d, g, s, w, l, m in SW if g == 'g')
wn_i = sum(w for t, d, g, s, w, l, m in INV if t == 'n')
wp_i = sum(w for t, d, g, s, w, l, m in INV if t == 'p')
rn = VDD / 2 / ext(nm, 'ID', 0.13, VDD, VDD / 2, 0.0, wn_i)
rp = VDD / 2 / ext(pm, 'ID', 0.13, VDD, VDD / 2, 0.0, wp_i)
t_inv = 0.69 * (rn + rp) / 2 * gb_load
print(f'inv_lv {wn_i:g}u / {wp_i:g}u drives {gb_load*1e15:.1f} fF of three switch gates, delay {t_inv*1e12:.0f} ps, against the {T_GAP*1e9:.2f} ns gap')
print(f'the phase pin itself drives {g_load*1e15:.1f} fF per phase')
print(f'a phase and its complement differ by {t_inv*1e12:.0f} ps at the switches, {t_inv/T_GAP*100:.0f} % of the gap')

inv_lv 1u / 2u drives 147.3 fF of three switch gates, delay 123 ps, against the 3.24 ns gap
the phase pin itself drives 147.3 fF per phase
a phase and its complement differ by 123 ps at the switches, 4 % of the gap


### Comparison with the drawing

In [10]:
n_set = int(np.ceil(c1r * ((1 - SETTLE[0] ** (1 / SETTLE[1])) / SETTLE[0] ** (1 / SETTLE[1])) / (C2u * REAL['refresh'])))
n_alt = int(np.ceil(c1r * ((1 - SETTLE_ALT[0] ** (1 / SETTLE_ALT[1])) / SETTLE_ALT[0] ** (1 / SETTLE_ALT[1])) / (C2u * REAL['refresh'])))
rows = [
 ('refresh C2, C4 (units)',   f'{n_set} or {n_alt}', N2, f'{SETTLE[1]} or {SETTLE_ALT[1]} cycles to 1 %, noise needs {np.ceil(u_noise):.0f}'),
 ('sense C1, C3 (units)',     'no spec',   N1, f'attenuation to vfb {2*c1r/(2*c1r+cpar):.2f}, the unit of the integrator caps'),
 ('switch Wn / Wp (um)',      f'{wn:g} / {3*wn:g}', f'{SWN:g} / {SWP:g}', 'Ron alone, the leakage and injection set the rest'),
 ('switch L (um)',            '1',      '1', 'leak limited, from sc_int2 and sw_tgate_cmfb2'),
 ('clock inverter',           'any',    f'{wn_i:g} / {wp_i:g}', f'{t_inv*1e12:.0f} ps against {T_GAP*1e9:.2f} ns'),
]
print(f"{'group':26s} {'derived':>10s} {'drawn':>10s}   basis")
for n, a, b, why in rows:
    print(f'{n:26s} {str(a):>10s} {str(b):>10s}   {why}')

group                         derived      drawn   basis
refresh C2, C4 (units)         3 or 4          4   20 or 15 cycles to 1 %, noise needs 2
sense C1, C3 (units)          no spec          3   attenuation to vfb 0.57, the unit of the integrator caps
switch Wn / Wp (um)         0.5 / 1.5      1 / 3   Ron alone, the leakage and injection set the rest
switch L (um)                       1          1   leak limited, from sc_int2 and sw_tgate_cmfb2
clock inverter                    any      1 / 2   123 ps against 3.24 ns


The derivation reproduces the drawn capacitor ratio through the settling of the common-mode loop (3 units for 20 cycles, 4 for 15, the drawn 4 units, k = 0.31 against the 31 % of the README), the sense unit (the integrator unit), the 1u/3u switch ratio and the L of 1u as consistent with the leakage. Switch Ron and the clock inverters leave a large margin against a phase of 485 ns and a gap of 3.24 ns.

Disagreements and open points, none of which changes a drawn size:

- The refresh cap is 4 units, one above the 20 cycle assumption (3 units). The cycle count is an assumption of this notebook, the README gives none.
- The switch is 2 times wider than Ron needs (1u/3u against 0.5u/1.5u). Leakage and injection decide that, and the leakage estimate is an upper bound at the table floor. Even that bound is 15 times inside a 20 mV common-mode error.
- The sense caps reach `vfb` through a divider of 0.57, because the OTA sinks put 148 fF on it (table estimate at L = 3 um, interpolated). No spec in the READMEs covers it.
- The README has no standalone simulation of the common-mode loop, so its settling is derived only.